In [1]:
import tensorflow as tf
from PIL import Image  # Import Pillow for image processing
import numpy as np
import time

class_labels_disease = ["bacterial_leaf_blight", "bacterial_leaf_streak", "bacterial_panicle_blight",
                        "blast", "brown_spot", "dead_heart", "downy_mildew", "hispa", "normal", "tungro"]

class_labels_varieties = ["ADT45", "AndraPonni", "AtchayaPonni",
                          "IR20", "KarnatakaPonni", "Onthanel", "Ponni", "RR", "Surya", "Zonal"]


In [2]:
def preprocess_image(img_path):
    img = Image.open(img_path)
    img = img.resize((224, 224))  # Resize to the target size
    img_array = np.array(img) / 255.0  # Normalize to [0, 1]
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    return img_array

In [3]:
def loader(model_name, model_path):
    print(f"Loading {model_name} model...")
    start_time = time.time()
    model = tf.keras.models.load_model(model_path)
    print(f"{model_name} model loaded in {time.time() - start_time:.2f} seconds.")
    return model

In [4]:
def load_all():
    global task1_model_vgg, task1_model_mobile_net, task1_model_cnn
    global task2_resnet50_model, task2_efficient_net_model, task2_model_cnn
    global task3_model_cnn

    # Load models
    task1_model_vgg = loader("VGG", "saved_model/Task1/task1_vgg_model.h5")
    task1_model_mobile_net = loader("MobileNet", "saved_model/Task1/task1_mobile_net_model.h5")
    task1_model_cnn = loader("CNN", "saved_model/Task1/task1_cnn_model.h5")

    task2_resnet50_model = loader("Resnet50", "saved_model/Task2/task2_resnet50_model.h5")
    task2_efficient_net_model = loader("EfficientNet", "saved_model/Task2/task2_efficient_net_model.h5")
    task2_model_cnn = loader("CNN", "saved_model/Task2/task2_cnn_model.h5")

    #task3_nasnet_model = loader("Nasnet", "saved_model/Task3/task3_nasnet_model.h5")
    task3_model_cnn = loader("CNN", "saved_model/Task3/task3_cnn_model.h5")

    # task3_nasnet_model = loader("Nasnet")
    # task3_model_cnn = loader("CNN")


In [5]:
load_all()

Loading VGG model...
VGG model loaded in 1.85 seconds.
Loading MobileNet model...
MobileNet model loaded in 3.18 seconds.
Loading CNN model...
CNN model loaded in 0.74 seconds.
Loading Resnet50 model...
Resnet50 model loaded in 3.99 seconds.
Loading EfficientNet model...
EfficientNet model loaded in 3.89 seconds.
Loading CNN model...
CNN model loaded in 1.06 seconds.
Loading CNN model...
CNN model loaded in 0.38 seconds.


In [6]:
def classify(file, file_path, mode):
    """
    Classifies the image using preloaded models based on the specified mode: 'disease' or 'variety'.

    Returns:
        dict: A dictionary of predictions from each model if successful.
        None: If an error occurs during processing.
    """
    img_array = preprocess_image(file_path)

    try:
        if mode == "disease":
            # Predict with disease models
            vgg_prediction = task1_model_vgg.predict(img_array)
            mobile_net_prediction = task1_model_mobile_net.predict(img_array)
            cnn_prediction = task1_model_cnn.predict(img_array)

            predictions = {
                "vgg": {
                    "label": class_labels_disease[int(np.argmax(vgg_prediction))],
                    "confidence": f"{float(np.max(vgg_prediction)) * 100:.2f}%",
                },
                "mobile_net": {
                    "label": class_labels_disease[int(np.argmax(mobile_net_prediction))],
                    "confidence": f"{float(np.max(mobile_net_prediction)) * 100:.2f}%",
                },
                "cnn": {
                    "label": class_labels_disease[int(np.argmax(cnn_prediction))],
                    "confidence": f"{float(np.max(cnn_prediction)) * 100:.2f}%",
                },
            }
            return predictions

        elif mode == "variety":
            # Predict with variety models
            resnet50_prediction = task2_resnet50_model.predict(img_array)
            efficient_net_prediction = task2_efficient_net_model.predict(img_array)
            cnn_prediction = task2_model_cnn.predict(img_array)

            predictions = {
                "resnet50": {
                    "label": class_labels_varieties[int(np.argmax(resnet50_prediction))],
                    "confidence": f"{float(np.max(resnet50_prediction)) * 100:.2f}%",
                },
                "efficient_net": {
                    "label": class_labels_varieties[int(np.argmax(efficient_net_prediction))],
                    "confidence": f"{float(np.max(efficient_net_prediction)) * 100:.2f}%",
                },
                "cnn": {
                    "label": class_labels_varieties[int(np.argmax(cnn_prediction))],
                    "confidence": f"{float(np.max(cnn_prediction)) * 100:.2f}%",
                },
            }
            return predictions
        elif mode == "age":
            #nasnet_prediction = task3_nasnet_model.predict(img_array)
            cnn_prediction = task3_model_cnn.predict(img_array)

            #nasnet_index = nasnet_prediction[0][0][0]

            cnn_index = cnn_prediction[0][0]

            predictions = {
                "cnn": {
                    "label": f"{cnn_index * 80:.2f}",
                },
            }
            return predictions
        else:
            print(f"Invalid classification mode: {mode}")
            return None

    except Exception as e:
        print(f"Error during classification ({mode}): {e}")
        return None


In [7]:
print(task1_model_vgg)
print(task2_resnet50_model)

In [ ]:
import pandas as pd
import os


csv_path = "prediction_submission.csv"
df = pd.read_csv(csv_path)
df['label'] = df['label'].astype(str)
df['variety'] = df['variety'].astype(str)
df['age'] = df['age'].astype(str)

for i, row in df.iterrows():
    image_id = row["image_id"]
    image_path = os.path.join("dataset","test_images", image_id)
    
    try:
        print(f"Classifying image {i+1}/{len(df)}: {image_id}")
        # Run classification
        disease_predictions = classify(image_path, image_path, mode="disease")
        variety_predictions = classify(image_path, image_path, mode="variety")
        age_predictions = classify(image_path, image_path, mode="age")
        # Find most confident disease prediction
        label = max(disease_predictions.items(), key=lambda x: float(x[1]['confidence'][:-1]))[1]['label']

        # Find most confident variety prediction
        variety = max(variety_predictions.items(), key=lambda x: float(x[1]['confidence'][:-1]))[1]['label']

        age = age_predictions["cnn"]["label"]

        # Update DataFrame
        df.at[i, 'label'] = label
        df.at[i, 'variety'] = variety
        df.at[i, 'age'] = age

    except Exception as e:
        print(f"Failed to classify {image_id}: {e}")
        continue

# Save updated CSV
df.to_csv(csv_path, index=False)


Classifying image 1/3469: 200001.jpg
1/1 [==============================] - 0s 311ms/step
Classifying image 2/3469: 200002.jpg
1/1 [==============================] - 0s 25ms/step
Classifying image 3/3469: 200003.jpg
1/1 [==============================] - 0s 23ms/step
Classifying image 4/3469: 200004.jpg
1/1 [==============================] - 0s 24ms/step
Classifying image 5/3469: 200005.jpg
1/1 [==============================] - 0s 25ms/step
Classifying image 6/3469: 200006.jpg
1/1 [==============================] - 0s 23ms/step
Classifying image 7/3469: 200007.jpg
1/1 [==============================] - 0s 30ms/step
Classifying image 8/3469: 200008.jpg
1/1 [==============================] - 0s 25ms/step
Classifying image 9/3469: 200009.jpg
1/1 [==============================] - 0s 23ms/step
Classifying image 10/3469: 200010.jpg
1/1 [==============================] - 0s 28ms/step
Classifying image 11/3469: 200011.jpg
1/1 [==============================] - 0s 25ms/step
Classifying image 